## Import libraries


In [ ]:
import os
from glob import glob

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio as rio
from shapely import wkb
from tqdm.auto import tqdm
from sklearn.manifold import TSNE
import matplotlib.patches as mpatches


# Helper function to format crop names for legend
def format_crop_name(crop_name):
    """Convert crop names to title case and remove underscores.
    Example: aman_rice -> Aman Rice"""
    return crop_name.replace("_", " ").title()


aef_paths = glob(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop_Classification/data/processed/AEF/*/*.tiff"
)

## Read the dataset


In [ ]:
# Read ROI
roi_gdf = gpd.read_file(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/raw/shapefile/west_bengal.gpkg"
)
roi_union = roi_gdf.unary_union

# Read index
df = pd.read_parquet(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop_Classification/data/raw/aef_index.parquet"
)
df = df[df["year"].isin([2023, 2024])]  # Filter for 2023 and 2024
df["geometry"] = df["geom"].apply(wkb.loads)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# Filter embeddings that intersect ROI
gdf_roi = gdf[gdf.geometry.intersects(roi_union)].copy()

print(f"Found {len(gdf_roi)} embeddings intersecting with ROI")

In [ ]:
# Read the crop samples
crop_samples = gpd.read_file(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/master/wbcrop_points.gpkg"
)
crop_samples["collection_date"] = pd.to_datetime(
    crop_samples["collection_date"], format="%Y-%m-%d"
)

# Extract the year
crop_samples["year"] = crop_samples["collection_date"].dt.year

# Split for 2023 and 2024
crop_samples_2023 = crop_samples[crop_samples["year"] == 2023].copy()
crop_samples_2024 = crop_samples[crop_samples["year"] == 2024].copy()

# Apply spatial join with the index file
crop_samples_2023 = gpd.sjoin(
    left_df=crop_samples_2023,
    right_df=gdf_roi[gdf_roi["year"] == 2023][["fid", "geometry"]],
    how="left",
    predicate="within",
)
crop_samples_2024 = gpd.sjoin(
    left_df=crop_samples_2024,
    right_df=gdf_roi[gdf_roi["year"] == 2024][["fid", "geometry"]],
    how="left",
    predicate="within",
)

crop_samples_2023.drop("index_right", axis=1, inplace=True)
crop_samples_2024.drop("index_right", axis=1, inplace=True)

print(crop_samples_2023.shape, crop_samples_2024.shape)
crop_samples_2023.head()

In [ ]:
# Plot the data
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))

roi_gdf.plot(ax=ax, facecolor="none", linewidth=0.5)
gdf_roi.plot(ax=ax, facecolor="none", edgecolor="r")
crop_samples.plot(ax=ax, column="crop", markersize=0.2)

plt.show()

## Extract the band values from the rasters


In [ ]:
crop_samples = {2023: crop_samples_2023, 2024: crop_samples_2024}

all_samples = []

for year in crop_samples.keys():
    print("-" * 50)
    print(" " * 22 + str(year) + " " * 22)

    data = crop_samples[year]

    for fid in tqdm(data["fid"].unique()):
        samples = data[data["fid"] == fid].copy()

        raster_path = f"/beegfs/halder/GITHUB/RESEARCH/WBCrop_Classification/data/processed/AEF/{year}/{fid}.tiff"

        with rio.open(raster_path) as src:
            samples = samples.to_crs(src.crs)

            # Ensure point geometry
            if samples.geometry.iloc[0].geom_type != "Point":
                samples["geometry"] = samples.geometry.centroid

            coords = [(geom.x, geom.y) for geom in samples.geometry]

            # Sample raster → NumPy array
            values = np.array(list(src.sample(coords)))  # shape: (n_points, n_bands)

            # Handle NoData (-128)
            values = values.astype(np.float32)
            values[values == -128] = np.nan

            # Dequantization
            values = ((values / 127.5) ** 2) * np.sign(values)

            # Convert to DataFrame
            band_names = [f"band_{i+1}" for i in range(src.count)]
            df_vals = pd.DataFrame(values, columns=band_names, index=samples.index)

        samples_df = pd.concat([samples[["id", "crop", "year"]], df_vals], axis=1)

        all_samples.append(samples_df)

# Final dataset
final_df = pd.concat(all_samples, ignore_index=True)

print(final_df.shape)
final_df.head()

In [ ]:
# Save the data
# final_df.to_parquet(
#     "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/processed/crop_samples.parquet",
#     index=False,
# )

## Apply T-SNE with two components


In [ ]:
# Read the data
data = pd.read_parquet(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/processed/crop_samples.parquet"
)
crop_samples = gpd.read_file(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/data/master/wbcrop_points.gpkg"
)

data = pd.merge(left=data, right=crop_samples[["id", "season"]], on="id", how="inner")
print(data.shape)
data.head()

In [ ]:
# Core seasonal splits
season_map = {
    "aman_rice": "kharif",
    "jute": "kharif",
    "mustard": "rabi",
    "vegetables": "rabi",
    "maize": "rabi",
    "flower": "rabi",
    "potato": "rabi",
    "wheat": "rabi",
    "boro_rice": "rabi",
    "tobacco": "rabi",
    "pine_apple": "perennial",
    "aus_rice": "zaid",
    "groundnut": "zaid",
    "betel_leaf": "perennial",
    "sugarcane": "perennial",
    "banana": "perennial",
    "others": "all",
    "tea": "perennial",
}
data["final_season"] = data["crop"].map(season_map)

# Core seasons
kharif_base = data[data["final_season"] == "kharif"]
rabi_base = data[data["final_season"] == "rabi"]
zaid_base = data[data["final_season"] == "zaid"]
perennial_base = data[data["final_season"] == "perennial"]

# Shared categories
shared_df = data[data["final_season"] == "all"]

kharif_df = pd.concat([kharif_base, shared_df], ignore_index=True)
rabi_df = pd.concat([rabi_base, shared_df], ignore_index=True)
zaid_df = pd.concat([zaid_base, shared_df], ignore_index=True)
perennial_df = pd.concat([perennial_base, shared_df], ignore_index=True)


for name, df_ in zip(
    ["kharif", "rabi", "zaid", "perennial"], [kharif_df, rabi_df, zaid_df, perennial_df]
):
    print(name)
    print(df_["season"].value_counts())
    print("-" * 30)

print(kharif_df.shape, rabi_df.shape, zaid_df.shape, perennial_df.shape)

In [ ]:
# Apply T-SNE to each seasonal dataset
from sklearn.manifold import TSNE

# Extract band features for T-SNE (exclude non-numeric columns)
band_cols = [col for col in kharif_df.columns if col.startswith("band_")]

# Dictionary to store T-SNE results
tsne_results = {}
tsne_fitted = {}

for season_name, season_df in [
    ("kharif", kharif_df),
    ("rabi", rabi_df),
    ("zaid", zaid_df),
    ("perennial", perennial_df),
]:
    print(f"\nProcessing {season_name}...")
    print(f"  Shape: {season_df.shape}")
    print(f"  Number of features: {len(band_cols)}")

    # Prepare data
    X = season_df[band_cols].values

    # Handle NaN values
    nan_mask = np.isnan(X).any(axis=1)
    print(f"  Samples with NaN: {nan_mask.sum()} / {len(X)}")

    X_clean = X[~nan_mask]
    indices_clean = season_df.index[~nan_mask]

    # Apply T-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, verbose=1)
    X_tsne = tsne.fit_transform(X_clean)

    # Store results
    tsne_results[season_name] = X_tsne
    tsne_fitted[season_name] = {
        "tsne_model": tsne,
        "X_clean": X_clean,
        "indices": indices_clean,
        "data": season_df.iloc[~nan_mask].copy(),
    }

    print(f"  T-SNE shape: {X_tsne.shape}")
    print(f"  KL divergence: {tsne.kl_divergence_:.2f}")

print("\nT-SNE computation complete!")

In [ ]:
# Plot T-SNE components for each season
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, (season_name, ax) in enumerate(
    zip(["kharif", "rabi", "zaid", "perennial"], axes)
):
    X_tsne = tsne_results[season_name]
    data = tsne_fitted[season_name]["data"]

    # Create scatter plot colored by crop
    crops = data["crop"].values
    unique_crops = np.unique(crops)

    # Create a color map
    cmap = plt.cm.get_cmap("tab10", len(unique_crops))
    colors_map = {crop: cmap(i) for i, crop in enumerate(unique_crops)}

    for crop in unique_crops:
        mask = crops == crop
        ax.scatter(
            X_tsne[mask, 0],
            X_tsne[mask, 1],
            alpha=0.4,
            s=10,
            label=format_crop_name(crop),
            color=colors_map[crop],
        )

    # ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("T-SNE Component 1")
    if idx == 0:
        ax.set_ylabel("T-SNE Component 2")
    else:
        ax.set_ylabel("")
    ax.set_title(f"{season_name.capitalize()} Season (n={len(X_tsne)})")
    ax.legend(
        bbox_to_anchor=(0.5, -0.7),
        ncol=2,
        loc="lower center",
        fontsize=12,
        markerscale=1,
        frameon=True,
        columnspacing=0.5,
    )
    ax.grid(True, alpha=0.3)

plt.savefig(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/output/figures/tsne_seasonal_plots.svg",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Check separability of different crops


In [ ]:
# Compute crop classification difficulty metrics
from sklearn.metrics import silhouette_samples, silhouette_score
from scipy.spatial.distance import cdist

crop_difficulty = {}

for season_name in ["kharif", "rabi", "zaid", "perennial"]:
    X_clean = tsne_fitted[season_name]["X_clean"]
    data = tsne_fitted[season_name]["data"]
    crops = data["crop"].values

    # Compute silhouette scores for each sample
    silhouette_vals = silhouette_samples(X_clean, crops)

    # Compute average silhouette score per crop
    crop_silhouettes = {}
    for crop in np.unique(crops):
        mask = crops == crop
        avg_silhouette = silhouette_vals[mask].mean()
        n_samples = mask.sum()
        crop_silhouettes[crop] = {
            "silhouette": avg_silhouette,
            "n_samples": n_samples,
            "std": silhouette_vals[mask].std(),
        }

    # Compute inter-crop distances (centroids)
    crop_centroids = {}
    for crop in np.unique(crops):
        mask = crops == crop
        centroid = X_clean[mask].mean(axis=0)
        crop_centroids[crop] = centroid

    # Find nearest neighbor crop for each crop (inter-class distance)
    crop_distances = {}
    for crop in np.unique(crops):
        distances = []
        for other_crop in np.unique(crops):
            if crop != other_crop:
                dist = np.linalg.norm(crop_centroids[crop] - crop_centroids[other_crop])
                distances.append(dist)
        crop_distances[crop] = min(distances) if distances else np.inf

    crop_difficulty[season_name] = {
        "silhouettes": crop_silhouettes,
        "distances": crop_distances,
        "data": data,
    }

    print(f"\n{season_name.upper()} SEASON - Classification Difficulty Analysis")
    print("=" * 60)

    # Sort by silhouette score
    sorted_crops = sorted(
        crop_silhouettes.items(), key=lambda x: x[1]["silhouette"], reverse=True
    )

    print("\nEASY TO CLASSIFY (High Silhouette Score):")
    for crop, metrics in sorted_crops[:5]:
        print(
            f"  {crop:20s}: Silhouette={metrics['silhouette']:6.3f} (n={metrics['n_samples']:5d})"
        )

    print("\nDIFFICULT TO CLASSIFY (Low Silhouette Score):")
    for crop, metrics in sorted_crops[-5:]:
        print(
            f"  {crop:20s}: Silhouette={metrics['silhouette']:6.3f} (n={metrics['n_samples']:5d})"
        )

In [ ]:
## Bar plots of crop difficulty by season
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, season_name in enumerate(["kharif", "rabi", "zaid", "perennial"]):
    silhouettes = crop_difficulty[season_name]["silhouettes"]

    # Sort by silhouette score
    sorted_crops = sorted(silhouettes.items(), key=lambda x: x[1]["silhouette"])
    crop_names = [format_crop_name(c[0]) for c in sorted_crops]
    scores = [c[1]["silhouette"] for c in sorted_crops]

    # Color: green for high (easy), red for low (difficult)
    colors = ["red" if s < -0.1 else "orange" if s < 0.1 else "green" for s in scores]

    ax = axes[idx]
    bars = ax.barh(crop_names, scores, color=colors, edgecolor="k", alpha=0.7)
    ax.set_xlabel("Silhouette Score", fontsize=11)
    ax.set_title(f"{season_name.capitalize()} Season", fontsize=12)
    ax.axvline(x=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.grid(True, alpha=0.3, axis="x")

    # Add value labels (kept inside plot area)
    x_min, x_max = ax.get_xlim()
    x_range = x_max - x_min

    for i, (bar, score) in enumerate(zip(bars, scores)):
        offset = 0.02 * x_range

        if score >= 0:
            x_pos = score - offset
            ha = "right"
        else:
            x_pos = score + offset
            ha = "left"

        # Clip text within axis limits
        x_pos = max(min(x_pos, x_max - offset), x_min + offset)

        ax.text(x_pos, i, f"{score:.2f}", va="center", ha=ha, fontsize=10)

plt.tight_layout()
plt.savefig(
    "/beegfs/halder/GITHUB/RESEARCH/WBCrop/output/figures/crop_difficulty_bars.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()